# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import re
import math
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("gutenberg", quiet=True)
file_ids = gutenberg.fileids()

raw_docs = []
titles = []
for f in file_ids:
    raw_docs.append(gutenberg.raw(f))
    titles.append(f.replace(".txt", ""))

def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    result = []
    for t in tokens:
        if len(t) > 2 and t not in ENGLISH_STOP_WORDS:
            result.append(t)
    return result

docs = []
doc_ids = []
for i in range(len(raw_docs)):
    docs.append(preprocess(raw_docs[i]))
    doc_ids.append("D" + str(i + 1))

unique_counts = []
lengths = []
all_terms = set()
for d in docs:
    unique_counts.append(len(set(d)))
    lengths.append(len(d))
    all_terms.update(d)

tabla = pd.DataFrame({
    "Identificador": doc_ids,
    "Título": titles,
    "Términos únicos": unique_counts,
})
print(tabla)

print("Número de documentos:", len(docs))
print("Tamaño del vocabulario:", len(all_terms))
print("Longitud mínima:", min(lengths))
print("Longitud media:", np.mean(lengths))
print("Longitud máxima:", max(lengths))

   Identificador               Título  Términos únicos
0             D1          austen-emma             6787
1             D2    austen-persuasion             5443
2             D3         austen-sense             5990
3             D4            bible-kjv            12290
4             D5          blake-poems             1328
5             D6       bryant-stories             3556
6             D7  burgess-busterbrown             1266
7             D8        carroll-alice             2298
8             D9      chesterton-ball             7855
9            D10     chesterton-brown             7423
10           D11  chesterton-thursday             5997
11           D12    edgeworth-parents             7977
12           D13   melville-moby_dick            16586
13           D14      milton-paradise             8693
14           D15   shakespeare-caesar             2805
15           D16   shakespeare-hamlet             4464
16           D17  shakespeare-macbeth             3217
17        

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
def compute_tf(doc):
    tf = {}
    for t in doc:
        tf[t] = tf.get(t, 0) + 1
    return tf

def compute_df(docs):
    df = {}
    for d in docs:
        for t in set(d):
            df[t] = df.get(t, 0) + 1
    return df

def compute_idf(df, n_docs):
    idf = {}
    for t in df:
        idf[t] = math.log(n_docs / df[t]) + 1
    return idf

def compute_tfidf(tf, idf):
    w = {}
    for t in tf:
        w[t] = tf[t] * idf[t]
    return w

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
N = len(docs)
vocab = sorted(all_terms)

term_index = {}
for i in range(len(vocab)):
    term_index[vocab[i]] = i

df = compute_df(docs)
idf = compute_idf(df, N)

idf_vec = np.zeros(len(vocab))
for t in vocab:
    idf_vec[term_index[t]] = idf[t]

X = np.zeros((N, len(vocab)))
for i in range(N):
    tf = compute_tf(docs[i])
    pesos = compute_tfidf(tf, idf)
    for t in pesos:
        X[i, term_index[t]] = pesos[t]

print("Matriz TF-IDF:", X.shape)

Matriz TF-IDF: (18, 41071)


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
def same_tokens(tokens):
    return tokens

vectorizer = TfidfVectorizer(
    analyzer=same_tokens,
    vocabulary=term_index,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
X_sk = vectorizer.fit_transform(docs)

diff = np.abs(X - X_sk.toarray())
print("Máxima diferencia absoluta:", diff.max())

Máxima diferencia absoluta: 0.0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [5]:
queries = {
    "Q1": "whale hunting ship sea captain",
    "Q2": "murder king throne betrayal",
    "Q3": "love marriage family estate",
    "Q4": "god lord faith commandments",
}
esperado = {
    "Q1": "Relatos de caza de ballenas y vida a bordo de un barco.",
    "Q2": "Tramas de poder, asesinato y traición en cortes reales.",
    "Q3": "Historias de matrimonio, familia y relaciones sociales.",
    "Q4": "Textos religiosos con mandamientos y fe en Dios.",
}

q_names = list(queries.keys())
q_tokens = []
for k in q_names:
    q_tokens.append(preprocess(queries[k]))

def query_vector(tokens):
    v = np.zeros(len(vocab))
    tf = compute_tf(tokens)
    for t in tf:
        if t in term_index:
            v[term_index[t]] = tf[t] * idf_vec[term_index[t]]
    return v

def cosine(q, d):
    denom = np.linalg.norm(q) * np.linalg.norm(d)
    if denom == 0:
        return 0.0
    return np.dot(q, d) / denom

Q = np.array([query_vector(t) for t in q_tokens])
Q_sk = vectorizer.transform(q_tokens).toarray()
print("Diferencia de vectores de consulta:", np.abs(Q - Q_sk).max())

S = np.zeros((len(Q), N))
for i in range(len(Q)):
    for j in range(N):
        S[i, j] = cosine(Q[i], X[j])

S_sk = cosine_similarity(Q_sk, X_sk)
print("Diferencia de similitudes:", np.abs(S - S_sk).max())

q = Q[0]
d = X[0]
compartidos = []
parciales = []
for t in set(q_tokens[0]):
    if t in term_index:
        i = term_index[t]
        if q[i] > 0 and d[i] > 0:
            compartidos.append(t)
            parciales.append(q[i] * d[i])

print("Consulta Q1 y documento D1")
print("Términos compartidos:", compartidos)
print("Productos parciales:", parciales)
print("Producto punto:", np.dot(q, d))
print("Norma consulta:", np.linalg.norm(q))
print("Norma documento:", np.linalg.norm(d))
print("Similitud:", cosine(q, d))

Diferencia de vectores de consulta: 0.0
Diferencia de similitudes: 1.3156142841808105e-14
Consulta Q1 y documento D1
Términos compartidos: ['hunting', 'captain', 'sea', 'ship']
Productos parciales: [np.float64(2.2274860585698666), np.float64(8.600242125114278), np.float64(17.492144807216793), np.float64(1.7567445395738914)]
Producto punto: 30.076617530474827
Norma consulta: 3.5361830306323023
Norma documento: 5604.827661792724
Similitud: 0.0015175115108434377


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
def fragment(text, terms, width=150):
    for t in terms:
        pos = text.lower().find(t)
        if pos != -1:
            inicio = max(0, pos - width // 2)
            return text[inicio:inicio + width].replace("\n", " ")
    return text[:width].replace("\n", " ")

for i in range(len(q_names)):
    orden = np.argsort(-S[i])[:5]
    filas = []
    for pos in range(5):
        j = orden[pos]
        filas.append({
            "Posición": pos + 1,
            "Identificador": doc_ids[j],
            "Título": titles[j],
            "Puntuación": S[i, j],
            "Fragmento": fragment(raw_docs[j], q_tokens[i]),
        })
    print(q_names[i], ":", queries[q_names[i]])
    print(pd.DataFrame(filas))
    print("Diferencia máxima con sklearn:", np.abs(S[i] - S_sk[i]).max())

Q1 : whale hunting ship sea captain
   Posición Identificador              Título  Puntuación  \
0         1           D13  melville-moby_dick    0.448980   
1         2            D2   austen-persuasion    0.101560   
2         3           D18      whitman-leaves    0.063902   
3         4           D10    chesterton-brown    0.032043   
4         5            D6      bryant-stories    0.027347   

                                           Fragmento  
0  While you take in hand to school others, and t...  
1  es of the case, only nodded in reply, and walk...  
2   king-pin, he heaves down with a strong arm, T...  
3  wn garments of Oberon, looked like a barbaric ...  
4  less somebody picked him out again.\r \r \r \r...  
Diferencia máxima con sklearn: 1.3156142841808105e-14
Q2 : murder king throne betrayal
   Posición Identificador               Título  Puntuación  \
0         1           D16   shakespeare-hamlet    0.038508   
1         2            D6       bryant-stories    0.0372